# 04 — Inference Engines: vLLM Profiling & Engine Selection

Experiments:
1. vLLM server configuration profiles (throughput / latency / balanced)
2. Async benchmark client measuring TTFT and TBT against a running vLLM server
3. Prefix caching: cold vs warm comparison
4. Engine selection decision function based on workload characteristics

In [ ]:
import sys, os, json, time, asyncio, statistics
from dataclasses import dataclass, field
from typing import Dict, List, Tuple

import aiohttp
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, '../..')
from utils.benchmark import BenchmarkResult
from utils.latency import LatencyTracker

## 1. vLLM Server Configuration Profiles

Three profiles optimizing for different objectives:
- **Throughput**: max batch size, continuous batching, chunked prefill
- **Latency**: small batch, eager mode, no chunked prefill
- **Balanced**: moderate batch with speculative decoding

In [ ]:
@dataclass
class VLLMProfile:
    name: str
    max_num_seqs: int
    max_num_batched_tokens: int
    enable_chunked_prefill: bool
    enable_prefix_caching: bool
    gpu_memory_utilization: float
    enforce_eager: bool
    speculative_model: str = None
    num_speculative_tokens: int = 0

    def to_cli_args(self) -> List[str]:
        args = [
            f"--max-num-seqs={self.max_num_seqs}",
            f"--max-num-batched-tokens={self.max_num_batched_tokens}",
            f"--gpu-memory-utilization={self.gpu_memory_utilization}",
        ]
        if self.enable_chunked_prefill:
            args.append("--enable-chunked-prefill")
        if self.enable_prefix_caching:
            args.append("--enable-prefix-caching")
        if self.enforce_eager:
            args.append("--enforce-eager")
        if self.speculative_model:
            args += [f"--speculative-model={self.speculative_model}",
                     f"--num-speculative-tokens={self.num_speculative_tokens}"]
        return args


PROFILES = {
    "throughput": VLLMProfile(
        name="throughput", max_num_seqs=256, max_num_batched_tokens=8192,
        enable_chunked_prefill=True, enable_prefix_caching=True,
        gpu_memory_utilization=0.95, enforce_eager=False),
    "latency": VLLMProfile(
        name="latency", max_num_seqs=8, max_num_batched_tokens=2048,
        enable_chunked_prefill=False, enable_prefix_caching=False,
        gpu_memory_utilization=0.85, enforce_eager=True),
    "balanced": VLLMProfile(
        name="balanced", max_num_seqs=64, max_num_batched_tokens=4096,
        enable_chunked_prefill=True, enable_prefix_caching=True,
        gpu_memory_utilization=0.90, enforce_eager=False,
        speculative_model="draft-model", num_speculative_tokens=5),
}

for name, p in PROFILES.items():
    print(f"\n=== {name.upper()} ===")
    print(f"  CLI: python -m vllm.entrypoints.openai.api_server {' '.join(p.to_cli_args())}")

## 2. Async Benchmark Client

Measures **Time To First Token (TTFT)** and **Time Between Tokens (TBT)** using streaming SSE from a vLLM OpenAI-compatible endpoint.

In [ ]:
VLLM_BASE_URL = os.getenv("VLLM_BASE_URL", "http://localhost:8000")

@dataclass
class RequestMetrics:
    ttft_ms: float = 0.0
    tbt_ms: List[float] = field(default_factory=list)
    total_ms: float = 0.0
    output_tokens: int = 0

    @property
    def mean_tbt_ms(self) -> float:
        return statistics.mean(self.tbt_ms) if self.tbt_ms else 0.0

    @property
    def p99_tbt_ms(self) -> float:
        return np.percentile(self.tbt_ms, 99) if self.tbt_ms else 0.0


async def benchmark_request(session: aiohttp.ClientSession, prompt: str,
                            max_tokens: int = 128, model: str = "default") -> RequestMetrics:
    """Send a single streaming request and measure TTFT/TBT."""
    payload = {"model": model, "prompt": prompt, "max_tokens": max_tokens,
               "stream": True, "temperature": 0.0}
    metrics = RequestMetrics()
    t_start = time.perf_counter()
    first_token = True
    last_token_time = t_start

    async with session.post(f"{VLLM_BASE_URL}/v1/completions", json=payload) as resp:
        async for line in resp.content:
            decoded = line.decode().strip()
            if not decoded.startswith("data:") or decoded == "data: [DONE]":
                continue
            now = time.perf_counter()
            if first_token:
                metrics.ttft_ms = (now - t_start) * 1000
                first_token = False
            else:
                metrics.tbt_ms.append((now - last_token_time) * 1000)
            last_token_time = now
            metrics.output_tokens += 1

    metrics.total_ms = (time.perf_counter() - t_start) * 1000
    return metrics


async def run_benchmark(prompts: List[str], concurrency: int = 8,
                        max_tokens: int = 128) -> List[RequestMetrics]:
    """Run concurrent benchmark requests."""
    sem = asyncio.Semaphore(concurrency)
    async with aiohttp.ClientSession() as session:
        async def bounded(p):
            async with sem:
                return await benchmark_request(session, p, max_tokens)
        return await asyncio.gather(*[bounded(p) for p in prompts])

print("Benchmark client ready. Set VLLM_BASE_URL to target a running server.")

In [ ]:
# Generate synthetic prompts of varying lengths
def make_prompts(n: int = 50, prefix: str = "Explain the concept of ") -> List[str]:
    topics = ["attention mechanisms", "KV caching", "tensor parallelism",
              "continuous batching", "speculative decoding", "quantization",
              "flash attention", "paged attention", "prefix caching",
              "disaggregated inference"]
    return [f"{prefix}{topics[i % len(topics)]} in detail." for i in range(n)]

# Example: run against live server (uncomment when server is up)
# results = await run_benchmark(make_prompts(50), concurrency=16, max_tokens=256)

# Simulated results for visualization demo
np.random.seed(42)
sim_results = [
    RequestMetrics(ttft_ms=np.random.lognormal(3.5, 0.4),
                   tbt_ms=list(np.random.lognormal(2.0, 0.3, size=128)),
                   total_ms=np.random.lognormal(6.5, 0.3), output_tokens=128)
    for _ in range(50)
]
print(f"Simulated {len(sim_results)} requests for visualization.")

In [ ]:
# Visualize TTFT and TBT distributions
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ttfts = [r.ttft_ms for r in sim_results]
axes[0].hist(ttfts, bins=20, color='steelblue', edgecolor='black', alpha=0.8)
axes[0].axvline(np.percentile(ttfts, 50), color='red', linestyle='--', label=f'P50={np.percentile(ttfts,50):.1f}ms')
axes[0].axvline(np.percentile(ttfts, 99), color='orange', linestyle='--', label=f'P99={np.percentile(ttfts,99):.1f}ms')
axes[0].set_xlabel('TTFT (ms)'); axes[0].set_ylabel('Count')
axes[0].set_title('Time To First Token'); axes[0].legend()

all_tbts = [t for r in sim_results for t in r.tbt_ms]
axes[1].hist(all_tbts, bins=40, color='coral', edgecolor='black', alpha=0.8)
axes[1].axvline(np.percentile(all_tbts, 50), color='red', linestyle='--', label=f'P50={np.percentile(all_tbts,50):.1f}ms')
axes[1].axvline(np.percentile(all_tbts, 99), color='orange', linestyle='--', label=f'P99={np.percentile(all_tbts,99):.1f}ms')
axes[1].set_xlabel('TBT (ms)'); axes[1].set_ylabel('Count')
axes[1].set_title('Time Between Tokens'); axes[1].legend()

plt.tight_layout(); plt.show()

## 3. Prefix Caching: Cold vs Warm

Measures TTFT improvement when the same system prompt prefix is reused across requests (KV cache hit).

In [ ]:
SYSTEM_PREFIX = "You are a helpful assistant specialized in machine learning infrastructure. " * 20  # ~400 tokens

async def prefix_cache_experiment(n_cold: int = 10, n_warm: int = 10,
                                  max_tokens: int = 64) -> Dict[str, List[float]]:
    """Run cold requests (unique prefixes) then warm requests (shared prefix)."""
    cold_ttfts, warm_ttfts = [], []
    async with aiohttp.ClientSession() as session:
        # Cold: each request has a unique prefix
        for i in range(n_cold):
            unique_prefix = f"Request {i} unique context: {os.urandom(64).hex()} "
            m = await benchmark_request(session, unique_prefix + "Summarize.", max_tokens)
            cold_ttfts.append(m.ttft_ms)

        # Warm: all requests share the same prefix
        for i in range(n_warm):
            m = await benchmark_request(session, SYSTEM_PREFIX + f"Question {i}: Explain.", max_tokens)
            warm_ttfts.append(m.ttft_ms)

    return {"cold": cold_ttfts, "warm": warm_ttfts}

# Uncomment with a running vLLM server (--enable-prefix-caching):
# cache_results = await prefix_cache_experiment(n_cold=20, n_warm=20)

# Simulated for visualization
cache_results = {
    "cold": list(np.random.lognormal(4.0, 0.3, size=20)),
    "warm": list(np.random.lognormal(3.2, 0.25, size=20)),
}
speedup = np.mean(cache_results['cold']) / np.mean(cache_results['warm'])
print(f"Cold mean TTFT: {np.mean(cache_results['cold']):.1f} ms")
print(f"Warm mean TTFT: {np.mean(cache_results['warm']):.1f} ms")
print(f"Prefix cache speedup: {speedup:.2f}x")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
positions = [1, 2]
bp = ax.boxplot([cache_results['cold'], cache_results['warm']], positions=positions,
                widths=0.5, patch_artist=True)
bp['boxes'][0].set_facecolor('#ffcccb')
bp['boxes'][1].set_facecolor('#90ee90')
ax.set_xticks(positions); ax.set_xticklabels(['Cold (unique prefix)', 'Warm (shared prefix)'])
ax.set_ylabel('TTFT (ms)'); ax.set_title(f'Prefix Caching Impact — {speedup:.2f}x Speedup')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

## 4. Engine Selection Decision Function

Given workload characteristics, select the optimal inference engine and configuration profile.

In [ ]:
@dataclass
class Workload:
    qps: float                    # queries per second
    avg_input_tokens: int         # average prompt length
    avg_output_tokens: int        # average generation length
    latency_sla_ms: float         # P99 TTFT SLA
    prefix_similarity: float      # 0-1, how much prompts share prefixes
    gpu_count: int = 1
    model_params_b: float = 7.0   # model size in billions


def select_engine(w: Workload) -> Dict[str, str]:
    """Select engine and profile based on workload characteristics."""
    # Memory check: rough estimate 2 bytes/param for fp16
    mem_required_gb = w.model_params_b * 2
    mem_per_gpu_gb = 80  # A100
    tp_degree = max(1, int(np.ceil(mem_required_gb / (mem_per_gpu_gb * 0.85))))

    # Engine selection logic
    if w.model_params_b > 70 and w.gpu_count >= 4:
        engine = "TensorRT-LLM"  # best multi-GPU throughput for very large models
    elif w.latency_sla_ms < 100 and w.qps < 10:
        engine = "vLLM (eager)"  # minimal overhead for strict latency
    elif w.prefix_similarity > 0.7:
        engine = "vLLM (prefix-caching)"  # exploit shared prefixes
    elif w.qps > 100:
        engine = "vLLM (chunked-prefill)"  # high throughput continuous batching
    else:
        engine = "vLLM (balanced)"

    # Profile selection
    if w.latency_sla_ms < 200:
        profile = "latency"
    elif w.qps > 50:
        profile = "throughput"
    else:
        profile = "balanced"

    return {
        "engine": engine,
        "profile": profile,
        "tp_degree": tp_degree,
        "reasoning": f"Model={w.model_params_b}B, QPS={w.qps}, SLA={w.latency_sla_ms}ms, "
                     f"prefix_sim={w.prefix_similarity:.0%} → {engine} / {profile}"
    }


# Test with different workloads
workloads = [
    Workload(qps=5, avg_input_tokens=2000, avg_output_tokens=100, latency_sla_ms=80,
             prefix_similarity=0.1, model_params_b=7),
    Workload(qps=200, avg_input_tokens=500, avg_output_tokens=256, latency_sla_ms=2000,
             prefix_similarity=0.3, model_params_b=13),
    Workload(qps=30, avg_input_tokens=1500, avg_output_tokens=200, latency_sla_ms=500,
             prefix_similarity=0.85, model_params_b=7),
    Workload(qps=10, avg_input_tokens=4000, avg_output_tokens=512, latency_sla_ms=3000,
             prefix_similarity=0.2, gpu_count=8, model_params_b=70),
]

for i, w in enumerate(workloads):
    result = select_engine(w)
    print(f"\nWorkload {i+1}: {result['reasoning']}")
    print(f"  → Engine: {result['engine']}, Profile: {result['profile']}, TP: {result['tp_degree']}")

In [ ]:
# Profile comparison: simulated throughput vs latency tradeoff
concurrency_levels = [1, 2, 4, 8, 16, 32, 64, 128]

# Simulated metrics per profile at each concurrency
def sim_profile_metrics(profile: str, conc: int) -> Tuple[float, float]:
    """Returns (throughput_tok_s, p99_ttft_ms) for a profile at given concurrency."""
    if profile == "throughput":
        tp = conc * 45 * (1 - 0.002 * conc)  # scales well, slight degradation
        lat = 30 + conc * 3.5
    elif profile == "latency":
        tp = min(conc, 8) * 60  # caps at max_num_seqs=8
        lat = 15 + conc * 0.5
    else:  # balanced
        tp = conc * 38 * (1 - 0.001 * conc)
        lat = 22 + conc * 2.0
    return (max(tp, 10), lat)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for profile, color in [("throughput", "blue"), ("latency", "red"), ("balanced", "green")]:
    metrics = [sim_profile_metrics(profile, c) for c in concurrency_levels]
    axes[0].plot(concurrency_levels, [m[0] for m in metrics], '-o', color=color, label=profile)
    axes[1].plot(concurrency_levels, [m[1] for m in metrics], '-o', color=color, label=profile)

axes[0].set_xlabel('Concurrency'); axes[0].set_ylabel('Throughput (tok/s)')
axes[0].set_title('Throughput vs Concurrency'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].set_xlabel('Concurrency'); axes[1].set_ylabel('P99 TTFT (ms)')
axes[1].set_title('Latency vs Concurrency'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

print('\n=== SUMMARY ===')
print('Profile configs: Throughput=256 seqs+chunked prefill, Latency=8 seqs+eager')
print('TTFT/TBT benchmark: Async streaming client captures per-token timing at scale')
print('Prefix caching: ~1.5-2x TTFT improvement for shared-prefix workloads')
print('Engine selection: Decision tree on QPS, SLA, model size, prefix similarity')